# 02 — SetSeq training and nested evaluation

One SetSeq model per unordered pair of left-out matches (21) and per outer fold (7). Each sees the frozen corruption plus two re-corruptions of its training matches, with random home/away mirroring. Decoder variants share the *same* predictions, so decoder ablations are paired exactly.

Then the augmentation ablation (`setseq_noaug`): same network, frozen corruption only, no mirroring.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp1_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_1')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 30)
import exp1_common as C
import exp1_protocol as P, exp1_net as N
print('device', N.DEVICE, '| config', N.TrainConfig().as_dict())

device cuda | config {'window': 300, 'batch': 32, 'epochs': 14, 'coverage': 2.0, 'lr': 0.002, 'weight_decay': 0.0001, 'width': 64, 'hidden': 64, 'boundary_weight': 0.5, 'boundary_pos_weight': 3.0, 'mirror': True, 'seed': 20261108}


## Fits (cached by code + config fingerprint)

In [2]:
t0 = time.time()
setseq_predictions = P.ensure_predictions('setseq')
print(f'total {time.time() - t0:.0f}s')

total 0s


## Decoder variants on identical predictions

In [3]:
for variant in ('setseq_bd', 'setseq_const', 'setseq_argmax'):
    t0 = time.time()
    result = P.evaluate_variant(variant, setseq_predictions)
    C.write_json(C.RESULTS / f'variant_{variant}.json', result)
    frame = pd.DataFrame(result['rows'])
    print(f'== {variant}  mean spellF1 {frame.spell_f1.mean():.4f}  ({time.time() - t0:.0f}s)')
    keep = [c for c in ['match', 'spell_f1', 'macro_f1', 'n_true', 'n_pred', 'gamma', 'lambda_team', 'lambda_dead', 'temperature'] if c in frame]
    print(frame[keep].round(4).to_string(index=False))
    if result['final_config']:
        print('final config (selected on 7 out-of-fold predictions, used for BAR-ATH):', result['final_config'])

== setseq_bd  mean spellF1 0.6556  (77s)
 match  spell_f1  macro_f1  n_true  n_pred  gamma  lambda_team  lambda_dead  temperature
J03WMX    0.6168    0.9196     511     336    0.0          1.0         16.0       1.2858
J03WN1    0.6439    0.9042     500     356    0.0          1.0          8.0       1.3308
J03WOH    0.6750    0.9203     402     318    0.0          0.5          1.0       1.2845
J03WOY    0.6657    0.9199     456     372    0.0          1.0          2.0       1.2853
J03WPY    0.6832    0.9273     421     343    0.0          1.0          4.0       1.2938
J03WQQ    0.6883    0.9211     401     383    0.0          1.0          8.0       1.2838
J03WR9    0.6162    0.9124     370     285    0.0          1.0          2.0       1.2738
final config (selected on 7 out-of-fold predictions, used for BAR-ATH): {'temperature': 1.3028667333854622, 'lambda_team': 0.5, 'lambda_dead': 0.5, 'gamma': 0.0}


== setseq_const  mean spellF1 0.6556  (18s)
 match  spell_f1  macro_f1  n_true  n_pred  gamma  lambda_team  lambda_dead  temperature
J03WMX    0.6168    0.9196     511     336    0.0          1.0         16.0       1.2858
J03WN1    0.6439    0.9042     500     356    0.0          1.0          8.0       1.3308
J03WOH    0.6750    0.9203     402     318    0.0          0.5          1.0       1.2845
J03WOY    0.6657    0.9199     456     372    0.0          1.0          2.0       1.2853
J03WPY    0.6832    0.9273     421     343    0.0          1.0          4.0       1.2938
J03WQQ    0.6883    0.9211     401     383    0.0          1.0          8.0       1.2838
J03WR9    0.6162    0.9124     370     285    0.0          1.0          2.0       1.2738
final config (selected on 7 out-of-fold predictions, used for BAR-ATH): {'temperature': 1.3028667333854622, 'lambda_team': 0.5, 'lambda_dead': 0.5, 'gamma': 0.0}
== setseq_argmax  mean spellF1 0.6240  (0s)
 match  spell_f1  macro_f1  n_true  n_

## Boundary head diagnostic (outer folds)
How well does p(boundary) rank true possession boundaries on held-out matches?

In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score
rows = []
for name in C.MATCH_IDS:
    _, boundary = setseq_predictions[f'outer_{name}'][name]
    target = P.load_tokens(name)['boundary']
    rows.append({'match': name, 'auc': roc_auc_score(target, boundary), 'avg_precision': average_precision_score(target, boundary), 'base_rate': target.mean()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

 match    auc  avg_precision  base_rate
J03WMX 0.9355         0.6070     0.0606
J03WN1 0.9395         0.6091     0.0613
J03WOH 0.9488         0.6098     0.0538
J03WOY 0.9514         0.6488     0.0586
J03WPY 0.9527         0.6295     0.0511
J03WQQ 0.9437         0.5724     0.0515
J03WR9 0.9488         0.5942     0.0455


## Augmentation ablation

In [5]:
t0 = time.time()
noaug_predictions = P.ensure_predictions('setseq_noaug')
result = P.evaluate_variant('setseq_noaug_bd', noaug_predictions)
C.write_json(C.RESULTS / 'variant_setseq_noaug_bd.json', result)
frame = pd.DataFrame(result['rows'])
print(f'setseq_noaug_bd mean spellF1 {frame.spell_f1.mean():.4f} ({time.time() - t0:.0f}s)')
print(frame[['match', 'spell_f1', 'macro_f1', 'n_true', 'n_pred', 'gamma']].round(4).to_string(index=False))

FIT setseq_noaug pair_J03WMX_J03WN1 14.8s


FIT setseq_noaug pair_J03WMX_J03WOH 14.1s


FIT setseq_noaug pair_J03WMX_J03WOY 14.1s


FIT setseq_noaug pair_J03WMX_J03WPY 14.1s


FIT setseq_noaug pair_J03WMX_J03WQQ 14.1s


FIT setseq_noaug pair_J03WMX_J03WR9 14.1s


... (19 outputs left out)

FIT setseq_noaug outer_J03WPY 16.6s


FIT setseq_noaug outer_J03WQQ 16.6s


FIT setseq_noaug outer_J03WR9 16.6s


setseq_noaug_bd mean spellF1 0.6005 (488s)
 match  spell_f1  macro_f1  n_true  n_pred  gamma
J03WMX    0.5748    0.9038     511     331    0.0
J03WN1    0.5914    0.8783     500     363    0.0
J03WOH    0.6058    0.9073     402     286    0.0
J03WOY    0.6258    0.9033     456     359    0.0
J03WPY    0.6514    0.9115     421     322    0.0
J03WQQ    0.6462    0.9065     401     356    0.0
J03WR9    0.5078    0.8550     370     320    0.0
